In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('exoplanets.csv', comment='#')

# Basic look
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 5 rows:\n", df.head())

# Check missing values
print("\nMissing values:\n", df.isnull().sum().sort_values(ascending=False).head(20))

# Basic stats
print("\nBasic Stats:\n", df.describe())

Shape: (6298, 84)

Columns:
 ['pl_name', 'hostname', 'sy_snum', 'sy_pnum', 'discoverymethod', 'disc_year', 'disc_facility', 'pl_controv_flag', 'pl_orbper', 'pl_orbpererr1', 'pl_orbpererr2', 'pl_orbperlim', 'pl_orbsmax', 'pl_orbsmaxerr1', 'pl_orbsmaxerr2', 'pl_orbsmaxlim', 'pl_rade', 'pl_radeerr1', 'pl_radeerr2', 'pl_radelim', 'pl_radj', 'pl_radjerr1', 'pl_radjerr2', 'pl_radjlim', 'pl_bmasse', 'pl_bmasseerr1', 'pl_bmasseerr2', 'pl_bmasselim', 'pl_bmassj', 'pl_bmassjerr1', 'pl_bmassjerr2', 'pl_bmassjlim', 'pl_bmassprov', 'pl_orbeccen', 'pl_orbeccenerr1', 'pl_orbeccenerr2', 'pl_orbeccenlim', 'pl_insol', 'pl_insolerr1', 'pl_insolerr2', 'pl_insollim', 'pl_eqt', 'pl_eqterr1', 'pl_eqterr2', 'pl_eqtlim', 'ttv_flag', 'st_spectype', 'st_teff', 'st_tefferr1', 'st_tefferr2', 'st_tefflim', 'st_rad', 'st_raderr1', 'st_raderr2', 'st_radlim', 'st_mass', 'st_masserr1', 'st_masserr2', 'st_masslim', 'st_met', 'st_meterr1', 'st_meterr2', 'st_metlim', 'st_metratio', 'st_logg', 'st_loggerr1', 'st_loggerr2',

In [5]:
# Keep only the useful columns
columns_needed = [
    'pl_name', 'hostname', 'discoverymethod', 'disc_year',
    'pl_orbper', 'pl_rade', 'pl_bmasse', 'pl_eqt',
    'sy_dist', 'sy_snum', 'sy_pnum'
]

df_clean = df[columns_needed].copy()

# Rename columns to readable names
df_clean.columns = [
    'planet_name', 'star_name', 'discovery_method', 'discovery_year',
    'orbital_period_days', 'planet_radius_earth', 'planet_mass_earth', 'equilibrium_temp_k',
    'distance_lightyears', 'num_stars', 'num_planets'
]

# Check missing values
print("Missing values before cleaning:")
print(df_clean.isnull().sum())

# Drop rows where key columns are missing
df_clean = df_clean.dropna(subset=['planet_radius_earth', 'discovery_year'])

# Fill remaining missing with median
df_clean['equilibrium_temp_k'] = df_clean['equilibrium_temp_k'].fillna(df_clean['equilibrium_temp_k'].median())
df_clean['planet_mass_earth'] = df_clean['planet_mass_earth'].fillna(df_clean['planet_mass_earth'].median())
df_clean['orbital_period_days'] = df_clean['orbital_period_days'].fillna(df_clean['orbital_period_days'].median())
df_clean['distance_lightyears'] = df_clean['distance_lightyears'].fillna(df_clean['distance_lightyears'].median())

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

print("\nClean dataset shape:", df_clean.shape)

# Save cleaned data
df_clean.to_csv('exoplanets_clean.csv', index=False)
print("\nSaved as exoplanets_clean.csv ✅")

Missing values before cleaning:
planet_name              0
star_name                0
discovery_method         0
discovery_year           1
orbital_period_days    340
planet_radius_earth     50
planet_mass_earth       31
equilibrium_temp_k     521
distance_lightyears     27
num_stars                0
num_planets              0
dtype: int64

Missing values after cleaning:
planet_name            0
star_name              0
discovery_method       0
discovery_year         0
orbital_period_days    0
planet_radius_earth    0
planet_mass_earth      0
equilibrium_temp_k     0
distance_lightyears    0
num_stars              0
num_planets            0
dtype: int64

Clean dataset shape: (6247, 11)

Saved as exoplanets_clean.csv ✅


In [ ]:
import pymysql
import pandas as pd

# Load clean data
df_clean = pd.read_csv('exoplanets_clean.csv')

# Connect to MySQL
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='your_password',  # 👈 change this
    database='nasa_exoplanets'
)

cursor = conn.cursor()

# Insert data row by row
for _, row in df_clean.iterrows():
    cursor.execute("""
        INSERT INTO exoplanets VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, tuple(row))

conn.commit()
conn.close()
print(f"Inserted {len(df_clean)} rows into MySQL ✅")

Inserted 6247 rows into MySQL ✅
